# Atividade Computacional: Fundamentos de Processamento de Imagens
**Unidade 1 | Capítulo 1 | Tarefa 2**
**Disciplina:** Visão Computacional — Residência em TIC43
**Prof.:** Alyson Bezerra

**Aluno(a):** Antonio Francisco Levi

Este notebook já vem com o **mini dataset próprio** (10 fotos tiradas com celular) organizado e todas as transformações da Parte 3 já processadas, prontas para análise.

> **Como usar:** faça upload do arquivo `mini_dataset_visao_computacional.zip` (enviado junto) na aba de arquivos do Colab e rode a célula de setup abaixo — ela descompacta tudo automaticamente.


## 0. Setup do ambiente

In [ ]:
import os, zipfile
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Descompacta o dataset (se ainda não estiver descompactado)
if not os.path.exists("dataset"):
    with zipfile.ZipFile("mini_dataset_visao_computacional.zip", "r") as z:
        z.extractall(".")

print("Pastas disponíveis:")
for root, dirs, files in os.walk("dataset"):
    print(root, "->", len(files), "arquivo(s)")


## Parte 1 — Definição das classes

- **Classe A:** Notebook (Acer, aberto e fechado, close no teclado)
- **Classe B:** Carregador (adaptador Acer e adaptador Samsung, plugues e cabos)

**Justificativa:** as duas classes são visualmente muito distintas — o notebook é um objeto grande, retangular, com teclado e tela; o carregador é pequeno, compacto, com cabo e plugue. A coleta foi fácil por serem itens do dia a dia disponíveis em casa.


## Parte 2 — Aquisição das imagens

As imagens já estão organizadas em `dataset/classe_A` (Notebook) e `dataset/classe_B` (Carregador), 5 fotos cada, totalizando 10.


In [ ]:
def carregar_imagens(pasta):
    caminhos = sorted([os.path.join(pasta, f) for f in os.listdir(pasta)
                        if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    return caminhos

caminhos_a = carregar_imagens("dataset/classe_A")
caminhos_b = carregar_imagens("dataset/classe_B")

fig, axs = plt.subplots(2, 5, figsize=(18, 7))
for i, cam in enumerate(caminhos_a):
    img = cv2.cvtColor(cv2.imread(cam), cv2.COLOR_BGR2RGB)
    axs[0, i].imshow(img)
    axs[0, i].set_title(os.path.basename(cam), fontsize=8)
    axs[0, i].axis("off")

for i, cam in enumerate(caminhos_b):
    img = cv2.cvtColor(cv2.imread(cam), cv2.COLOR_BGR2RGB)
    axs[1, i].imshow(img)
    axs[1, i].set_title(os.path.basename(cam), fontsize=8)
    axs[1, i].axis("off")

plt.suptitle("Classe A - Notebook (linha 1)  |  Classe B - Carregador (linha 2)")
plt.tight_layout()
plt.show()


### Registro técnico da aquisição

| Imagem | Classe | Dispositivo | Condição de iluminação | Distância/ângulo | Observação |
|---|---|---|---|---|---|
| img001_original | A | Motorola Moto G24 (câmera traseira) | luz artificial interna | close, ângulo lateral | leve desfoque de movimento |
| img002_original | A | Motorola Moto G24 | luz artificial interna | plano geral, de cima | nítida |
| img003_original | A | Motorola Moto G24 | luz artificial interna | close, outro ângulo do teclado | leve desfoque |
| img004_original | A | Motorola Moto G24 | luz ambiente/sombra | plano geral, de cima, notebook fechado | nítida |
| img005_original | A | Motorola Moto G24 | ambiente escuro, tela ligada (luz da própria tela) | plano geral, ângulo baixo | mais escura, contraste alto |
| img006_original | B | Motorola Moto G24 | luz artificial interna | close, vista frontal do carregador Acer | nítida |
| img007_original | B | Motorola Moto G24 | luz natural (externa) | plano próximo, carregador Samsung na tomada | boa nitidez |
| img008_original | B | Motorola Moto G24 | luz natural | close extremo no plugue/tomada | desfocada |
| img009_original | B | Motorola Moto G24 | luz artificial interna | close extremo no cabo/entrada do notebook | nítida |
| img010_original | B | Motorola Moto G24 | luz artificial interna | outro ângulo do carregador Acer | leve desfoque |

**Dispositivo utilizado:** Motorola Moto G24, câmera traseira principal.

**Observação geral sobre a aquisição:** o conjunto atende aos requisitos mínimos — há pelo menos 2 condições de iluminação (luz artificial interna predominando, mas também luz natural nas imagens 007 e 008, e ambiente escuro com luz de tela na 005), e pelo menos 2 ângulos/distâncias diferentes em cada classe (planos gerais e closes). Algumas fotos (001, 003, 008, 010) apresentam leve desfoque de movimento, típico de fotos tiradas à mão sem apoio — isso é discutido no impacto técnico mais abaixo.


## Parte 3 — Transformações de Imagens

Usamos **img001** (Classe A - Notebook) e **img006** (Classe B - Carregador) como exemplos para todas as transformações abaixo.


In [ ]:
exemplo_a = "dataset/classe_A/img001_original.jpg"
exemplo_b = "dataset/classe_B/img006_original.jpg"

img_a = cv2.cvtColor(cv2.imread(exemplo_a), cv2.COLOR_BGR2RGB)
img_b = cv2.cvtColor(cv2.imread(exemplo_b), cv2.COLOR_BGR2RGB)

print("Exemplo Classe A:", exemplo_a, "| shape:", img_a.shape)
print("Exemplo Classe B:", exemplo_b, "| shape:", img_b.shape)

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(img_a); axs[0].set_title("Exemplo Classe A - Notebook"); axs[0].axis("off")
axs[1].imshow(img_b); axs[1].set_title("Exemplo Classe B - Carregador"); axs[1].axis("off")
plt.show()


### A. Análise de Resolução

In [ ]:
def gerar_versoes_resolucao(img, nome_classe):
    h, w = img.shape[:2]
    versoes = {
        "100% (original)": img,
        "50%": cv2.resize(img, (int(w*0.5), int(h*0.5)), interpolation=cv2.INTER_AREA),
        "20%": cv2.resize(img, (int(w*0.2), int(h*0.2)), interpolation=cv2.INTER_AREA),
    }
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (nome, im) in zip(axs, versoes.items()):
        ax.imshow(im)
        ax.set_title(f"{nome_classe} - {nome}\n{im.shape[1]}x{im.shape[0]}px")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return versoes

versoes_res_a = gerar_versoes_resolucao(img_a, "Classe A - Notebook")
versoes_res_b = gerar_versoes_resolucao(img_b, "Classe B - Carregador")


**Comentário sobre perda de detalhe e impacto em Visão Computacional:**

Na resolução original (1280x963), dá para ler claramente os símbolos do teclado e o logo "acer" no carregador. Na versão de 50% (640x481) a perda é pequena — ainda é possível reconhecer os textos e ícones, embora um pouco mais suaves. Já na versão de 20% (256x192) a perda é significativa: as letras do teclado praticamente desaparecem e o logo do carregador fica borrado, quase ilegível. Isso mostra que, para tarefas que dependem de **detalhes finos** (leitura de texto, reconhecimento de pequenos ícones, detecção de bordas finas), reduzir demais a resolução compromete o resultado. Para tarefas de classificação mais geral (reconhecer "é um notebook" vs "é um carregador"), mesmo 20% ainda preserva a forma geral do objeto, então pode ser aceitável e economiza bastante processamento.


### B. Espaço de Cor (RGB, HSV, Escala de Cinza)

In [ ]:
def gerar_espacos_cor(img, nome_classe):
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Separa os 3 canais do HSV para visualizacao individual (mais didatico que mostrar o HSV bruto)
    h_canal, s_canal, v_canal = cv2.split(hsv)

    fig, axs = plt.subplots(1, 5, figsize=(22, 5))
    axs[0].imshow(img); axs[0].set_title(f"{nome_classe} - RGB"); axs[0].axis("off")
    axs[1].imshow(h_canal, cmap="hsv"); axs[1].set_title(f"{nome_classe} - HSV: canal H (matiz)"); axs[1].axis("off")
    axs[2].imshow(s_canal, cmap="gray"); axs[2].set_title(f"{nome_classe} - HSV: canal S (saturacao)"); axs[2].axis("off")
    axs[3].imshow(v_canal, cmap="gray"); axs[3].set_title(f"{nome_classe} - HSV: canal V (brilho)"); axs[3].axis("off")
    axs[4].imshow(gray, cmap="gray"); axs[4].set_title(f"{nome_classe} - Escala de cinza"); axs[4].axis("off")
    plt.tight_layout()
    plt.show()
    return hsv, gray

hsv_a, gray_a = gerar_espacos_cor(img_a, "Classe A - Notebook")
hsv_b, gray_b = gerar_espacos_cor(img_b, "Classe B - Carregador")


**Registro textual:**

- **Quais diferenças visuais aparecem ao converter?** Como o notebook e o carregador têm cores predominantemente neutras (preto, cinza, marrom da mesa), a diferença entre RGB e escala de cinza é sutil — a cor não é uma característica muito discriminante nessas fotos. Ao separar o HSV em seus 3 canais, fica mais claro o que cada um representa: o canal H (matiz) mostra pouca variação, já que os objetos são majoritariamente acinzentados/pretos; o canal S (saturação) fica bem escuro pelo mesmo motivo (cores pouco saturadas); e o canal V (brilho) é o que mais se parece com a escala de cinza, concentrando a maior parte da informação visual útil da cena.
- **Em que tipo de tarefa você usaria cada uma?**
  - **RGB:** quando a cor real do objeto importa para diferenciar classes (não é o caso ideal aqui, já que ambos objetos são escuros).
  - **HSV:** útil se eu quisesse segmentar objetos por cor de forma mais robusta à iluminação — nesse dataset, o canal V isolado já ajudaria bastante, pois concentra a informação de forma/contraste com pouca influência de matiz e saturação.
  - **Escala de cinza:** adequada para este dataset, já que a distinção entre notebook e carregador está mais na **forma e textura** (teclado com grade de teclas x superfície lisa do carregador) do que na cor — reduz custo computacional sem perder a informação relevante.


### C. Quantização

In [ ]:
def quantizar(img_gray, niveis):
    fator = 256 // niveis
    quantizada = (img_gray // fator) * fator
    return quantizada.astype(np.uint8)

def gerar_quantizacoes(img_gray, nome_classe):
    niveis_lista = [256, 64, 32, 2]
    versoes = {}
    fig, axs = plt.subplots(1, 4, figsize=(18, 5))
    for ax, n in zip(axs, niveis_lista):
        im_q = img_gray if n == 256 else quantizar(img_gray, n)
        versoes[n] = im_q
        ax.imshow(im_q, cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"{nome_classe} - {n} níveis")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return versoes

quant_a = gerar_quantizacoes(gray_a, "Classe A - Notebook")
quant_b = gerar_quantizacoes(gray_b, "Classe B - Carregador")


**Registro:**

- **O que muda visualmente?** Entre 256 e 64 níveis praticamente não há diferença perceptível a olho nu. Com 32 níveis já aparecem leves "degraus" de tom em áreas de transição suave (como o reflexo de luz no carregador). Com apenas 2 níveis, a imagem vira puro preto e branco: no notebook ainda dá pra reconhecer o contorno das teclas como manchas cinza/pretas (a silhueta geral se mantém), mas todo o detalhe interno de textura desaparece; no carregador só sobra o contorno do objeto contra o fundo.
- **Qual versão ainda seria mais útil para o seu dataset? Para qual aplicação?** A versão de **64 níveis** parece o melhor equilíbrio: reduz a quantidade de informação (e portanto o tamanho/custo) sem prejudicar a percepção visual do objeto. Já a versão de **2 níveis** só seria útil para uma tarefa bem específica, como segmentação binária de silhueta/contorno (separar "objeto" de "fundo"), não para classificar entre notebook e carregador, pois a essa altura muita informação discriminante se perde.


### D. Formato de arquivo (JPEG x PNG)

In [ ]:
def salvar_formatos(img_rgb, nome_classe):
    pil_img = Image.fromarray(img_rgb)
    caminho_jpeg = f"transformacoes/{nome_classe}_formato.jpg"
    caminho_png = f"transformacoes/{nome_classe}_formato.png"
    os.makedirs("transformacoes", exist_ok=True)
    pil_img.save(caminho_jpeg, "JPEG", quality=70)
    pil_img.save(caminho_png, "PNG")

    tam_jpeg = os.path.getsize(caminho_jpeg) / 1024
    tam_png = os.path.getsize(caminho_png) / 1024
    print(f"[{nome_classe}] JPEG: {tam_jpeg:.1f} KB  |  PNG: {tam_png:.1f} KB  (PNG é {tam_png/tam_jpeg:.1f}x maior)")

    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    axs[0].imshow(Image.open(caminho_jpeg)); axs[0].set_title(f"JPEG ({tam_jpeg:.1f} KB)"); axs[0].axis("off")
    axs[1].imshow(Image.open(caminho_png)); axs[1].set_title(f"PNG ({tam_png:.1f} KB)"); axs[1].axis("off")
    plt.tight_layout()
    plt.show()
    return tam_jpeg, tam_png

tam_jpeg_a, tam_png_a = salvar_formatos(img_a, "classe_A")
tam_jpeg_b, tam_png_b = salvar_formatos(img_b, "classe_B")


**Registro:**

- **Tamanho do arquivo:** na Classe A (Notebook), JPEG ficou com ~47 KB contra ~501 KB do PNG (PNG cerca de 10,6x maior). Na Classe B (Carregador), JPEG ficou com ~85 KB contra ~968 KB do PNG (PNG cerca de 11,5x maior).
- **Diferença visual percebida:** a olho nu, com qualidade JPEG 70, não há diferença perceptível entre as duas versões nessas fotos — os detalhes visíveis (textura da mesa, logo do carregador) parecem preservados.
- **Hipótese sobre impacto em tarefas de PDI:** para este dataset, como as fotos já têm ruído/desfoque naturais da captura com celular, a perda adicional do JPEG é irrelevante perto da perda de qualidade original — então **JPEG é a escolha mais eficiente**, economizando bastante espaço sem prejuízo perceptível. PNG só valeria a pena se fôssemos fazer anotação pixel a pixel (ex.: segmentação semântica), onde qualquer artefato de compressão pode interferir na anotação exata das bordas.


## Parte 4 — Estrutura do Mini Dataset

O dataset já está organizado no padrão de ML:

```
dataset/
  classe_A/
    img001_original.jpg
    img002_original.jpg
    img003_original.jpg
    img004_original.jpg
    img005_original.jpg
  classe_B/
    img006_original.jpg
    img007_original.jpg
    img008_original.jpg
    img009_original.jpg
    img010_original.jpg
```


In [ ]:
print("Estrutura final do dataset:\n")
for root, dirs, filenames in os.walk("dataset"):
    nivel = root.replace("dataset", "").count(os.sep)
    indent = " " * 2 * nivel
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (nivel + 1)
    for f in sorted(filenames):
        print(f"{subindent}{f}")


In [ ]:
import zipfile
with zipfile.ZipFile("mini_dataset_final.zip", "w") as zipf:
    for pasta in ["dataset", "transformacoes"]:
        for root, dirs, filenames in os.walk(pasta):
            for f in filenames:
                caminho_completo = os.path.join(root, f)
                zipf.write(caminho_completo)

print("Arquivo 'mini_dataset_final.zip' criado com sucesso (dataset original + transformações).")
# from google.colab import files
# files.download("mini_dataset_final.zip")


## Conclusão / Análise Técnica Final

- **Resolução:** para este dataset, resoluções muito baixas (20%) já comprometem a leitura de detalhes finos (texto, logotipos), mas mantêm a forma geral reconhecível. Para um classificador simples "notebook x carregador", uma resolução intermediária (50%) já seria suficiente e mais barata computacionalmente do que manter a resolução original.
- **Espaço de cor:** como as duas classes se diferenciam mais pela **forma** (grade de teclas x bloco liso) do que pela cor, a escala de cinza é a opção mais eficiente — reduz dimensionalidade sem perder a informação discriminante relevante.
- **Quantização:** reduzir para 64 níveis de cinza não prejudica perceptivelmente o dataset; já 2 níveis eliminaria informação de textura importante para diferenciar as classes, sendo útil só para tarefas de segmentação de silhueta.
- **Formato de arquivo:** JPEG é a escolha mais racional aqui — o ganho de espaço é grande (~10x menor que PNG) e a perda de qualidade é imperceptível dado que as fotos originais já têm limitações de nitidez (fotos tiradas à mão, sem tripé).
- **Reflexão geral:** em um projeto real (industrial, médico ou científico), essas mesmas decisões (resolução, espaço de cor, quantização, formato) teriam impacto direto no custo de armazenamento, velocidade de treinamento e até na acurácia do modelo. Aqui, como as classes são visualmente muito distintas em forma, o dataset tolera bem reduções de resolução e cor — mas em cenários com classes mais parecidas entre si, cada uma dessas escolhas precisaria ser bem mais conservadora para não descartar a informação que realmente diferencia as classes.
